# Manual Input Verification

Displays the four `AircraftCommand` channels (`n_z`, `n_y`, `roll_rate`, `throttle`)
in real time from a physical joystick.

**All input logic runs in C++.** This notebook launches `joystick_verify` as a subprocess
and reads the JSON lines it streams to stdout. No axis pipeline, dead zone, calibration,
trim, or command limiting is implemented here.

Design authority: [`docs/architecture/manual_input.md`](../docs/architecture/manual_input.md),
section *Verification Notebook*.

---

## Prerequisites

1. Build the project (`mingw32-make -C build`) so `build/tools/joystick_verify.exe` exists.
2. Connect a joystick before running the setup cell.
3. Install Python dependencies: `pip install PySide6 matplotlib`.

---

## Imports

In [ ]:
import json
import math
import queue
import subprocess
import sys
import threading
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.figure as mpl_figure
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg
from PySide6.QtCore import QTimer
from PySide6.QtWidgets import QApplication, QLabel, QMainWindow, QVBoxLayout, QWidget

---
## Configuration

Edit these values to match your build location and device.
All axis mapping, dead zone, and calibration options are passed as flags to
`joystick_verify` and handled in C++.

In [ ]:
# Path to the joystick_verify executable (relative to this notebook).
JOYSTICK_VERIFY = Path("../build/tools/joystick_verify.exe")

# Device selection: set DEVICE_NAME to a substring of the device name for
# name-based selection, or set DEVICE_INDEX for index-based selection.
# DEVICE_NAME takes priority when non-empty.
DEVICE_NAME  = ""   # e.g. "Logitech" — empty means use DEVICE_INDEX
DEVICE_INDEX = 0

# Optional path to a JSON config file for axis mapping, dead zone, etc.
# Leave empty to use joystick_verify defaults.
CONFIG_FILE = ""

# Polling rate in Hz (must match or be slower than joystick_verify --rate-hz).
RATE_HZ = 50.0

# Display constants — must match the C++ JoystickInputConfig defaults
# (or your config file overrides) so crosshairs and gauge neutrals are correct.
IDLE_THROTTLE_ND   = 0.05   # idle_throttle_nd default
MIN_NZ_G           = -2.0
MAX_NZ_G           =  4.0
MAX_NY_G           =  2.0
MAX_ROLL_RATE_RADS =  math.pi / 2.0   # 1.5708 rad/s default

---
## Device Enumeration

Run this cell to launch `joystick_verify` briefly and list detected joystick devices.
Adjust `DEVICE_NAME` or `DEVICE_INDEX` above before opening the display window.

In [ ]:
exe = JOYSTICK_VERIFY.resolve()
if not exe.exists():
    raise FileNotFoundError(
        f"joystick_verify not found at {exe}\n"
        "Build the project first: mingw32-make -C build"
    )

# Launch briefly just to read the device table (lines before READY).
proc = subprocess.Popen(
    [str(exe), "--device-index", str(DEVICE_INDEX)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

devices = []
for line in proc.stdout:
    line = line.rstrip()
    if line == "READY":
        break
    if line.startswith("DEVICE "):
        # Format: DEVICE <index> <num_axes> <name>
        parts = line.split(" ", 3)
        devices.append({"index": int(parts[1]), "axes": int(parts[2]), "name": parts[3]})

proc.terminate()
proc.wait()

print(f"Detected {len(devices)} joystick device(s):")
for d in devices:
    marker = " <-- selected" if d["index"] == DEVICE_INDEX and not DEVICE_NAME else ""
    print(f"  [{d['index']}]  {d['name']}  ({d['axes']} axes){marker}")
if not devices:
    print("  No devices found. Connect a joystick and re-run this cell.")

---
## Monitor Window

Defines the Qt display window. `joystick_verify` is launched as a background subprocess.
A reader thread puts each JSON line into a queue; the 20 Hz QTimer drains the queue,
takes the most recent frame, and redraws the display.

In [ ]:
def _build_args() -> list[str]:
    """Build the joystick_verify command-line arguments from configuration."""
    args = [str(JOYSTICK_VERIFY.resolve()), "--rate-hz", str(RATE_HZ)]
    if DEVICE_NAME:
        args += ["--device-name", DEVICE_NAME]
    else:
        args += ["--device-index", str(DEVICE_INDEX)]
    if CONFIG_FILE:
        args += ["--config", CONFIG_FILE]
    return args


def _stdout_reader(proc: subprocess.Popen, q: queue.Queue, ready: threading.Event) -> None:
    """Background thread: read lines from joystick_verify stdout into a queue."""
    for line in proc.stdout:
        line = line.rstrip()
        if line == "READY":
            ready.set()
            continue
        if line.startswith("DEVICE ") or not line:
            continue
        q.put(line)


def _draw_gauge(
    ax,
    value: float,
    v_min: float,
    v_max: float,
    v_neutral: float,
    label: str,
    unit: str,
) -> None:
    """Draw one horizontal bar gauge. Bar extends from neutral to current value."""
    ax.clear()
    span = v_max - v_min
    ax.barh(0, span, left=v_min, height=0.55, color="#ebebeb", edgecolor="#c8c8c8", linewidth=0.5)

    half_span = span / 2.0
    deflection = abs(value - v_neutral) / half_span if half_span > 0 else 0.0
    if deflection < 0.25:
        bar_color = "#1565C0"
    elif deflection < 0.75:
        bar_color = "#FF9800"
    else:
        bar_color = "#F44336"

    ax.barh(0, value - v_neutral, left=v_neutral, height=0.55, color=bar_color, alpha=0.85)
    ax.axvline(v_neutral, color="#303030", linewidth=1.5, zorder=4)
    ax.text(
        0.99, 0.5, f"{value:+6.2f} {unit}",
        transform=ax.transAxes, va="center", ha="right",
        fontsize=10, fontfamily="monospace", fontweight="bold",
    )
    ax.set_title(label, fontsize=9, pad=3, loc="left", fontweight="bold", color="#404040")
    ax.set_xlim(v_min, v_max)
    ax.set_ylim(-0.9, 0.9)
    ax.set_yticks([])
    ax.tick_params(axis="x", labelsize=7)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)


class InputMonitorWindow(QMainWindow):
    """Qt window showing live AircraftCommand values from joystick_verify.

    Layout:
      Row 1: two 2D box plots (Throttle/n_y  and  n_z/Roll Rate)
      Row 2: four horizontal bar gauges (n_z, n_y, roll rate, throttle)

    The window owns the joystick_verify subprocess and terminates it on close.
    """

    UPDATE_MS = 50  # 20 Hz display update

    def __init__(self) -> None:
        super().__init__()
        self.setWindowTitle("Manual Input Verification — LiteAero Sim")
        self.resize(900, 560)

        # --- Subprocess + reader thread ---
        self._q: queue.Queue = queue.Queue()
        self._ready = threading.Event()
        self._proc = subprocess.Popen(
            _build_args(),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        self._reader = threading.Thread(
            target=_stdout_reader,
            args=(self._proc, self._q, self._ready),
            daemon=True,
        )
        self._reader.start()

        # Wait for joystick_verify to print the READY sentinel before starting the timer.
        if not self._ready.wait(timeout=5.0):
            self._proc.terminate()
            raise RuntimeError("joystick_verify did not become ready within 5 s.")

        # --- Matplotlib figure ---
        self._fig = mpl_figure.Figure(figsize=(10, 5.5), facecolor="#f5f5f5")
        gs = gridspec.GridSpec(
            2, 4,
            figure=self._fig,
            left=0.06, right=0.97, top=0.93, bottom=0.08,
            hspace=0.55, wspace=0.35,
        )
        # Row 1: two 2D boxes, each spanning 2 columns
        self._ax_box_left  = self._fig.add_subplot(gs[0, 0:2])
        self._ax_box_right = self._fig.add_subplot(gs[0, 2:4])
        # Row 2: four bar gauges
        self._ax_gauges = [self._fig.add_subplot(gs[1, i]) for i in range(4)]

        self._init_boxes()

        self._canvas = FigureCanvasQTAgg(self._fig)

        # --- Layout ---
        central = QWidget()
        layout = QVBoxLayout(central)
        layout.setContentsMargins(4, 4, 4, 4)
        layout.addWidget(self._canvas)
        self.setCentralWidget(central)

        self._status = QLabel("Waiting for frames…")
        self._status.setStyleSheet("padding-left: 6px; font-size: 11px;")
        self.statusBar().addWidget(self._status)

        # --- Timer ---
        self._timer = QTimer(self)
        self._timer.timeout.connect(self._tick)
        self._timer.start(self.UPDATE_MS)

        # Neutral frame as initial display state
        self._last_frame = {
            "n_z": 1.0, "n_y": 0.0,
            "roll_rate_wind_rps": 0.0,
            "throttle_nd": IDLE_THROTTLE_ND,
            "actions": 0, "connected": False,
        }

    def _init_boxes(self) -> None:
        """Set up the static elements of the two 2D box plots."""
        # Left box: Throttle (Y, up) vs n_y (X, positive right)
        ax = self._ax_box_left
        ax.set_title("Throttle / n_y", fontsize=10, fontweight="bold", color="#404040")
        ax.set_xlabel("n_y  (g)", fontsize=8)
        ax.set_ylabel("Throttle", fontsize=8)
        ax.set_xlim(-MAX_NY_G, MAX_NY_G)
        ax.set_ylim(0.0, 1.0)
        ax.axvline(0.0, color="#aaaaaa", linewidth=0.8)
        ax.axhline(IDLE_THROTTLE_ND, color="#aaaaaa", linewidth=0.8)
        ax.tick_params(labelsize=7)
        ax.set_facecolor("#fafafa")
        self._scatter_left = ax.scatter(
            [0.0], [IDLE_THROTTLE_ND],
            s=80, color="#1565C0", zorder=5, clip_on=True,
        )

        # Right box: n_z (Y, positive DOWN) vs roll rate deg/s (X, positive right)
        ax = self._ax_box_right
        max_roll_deg = math.degrees(MAX_ROLL_RATE_RADS)
        ax.set_title("n_z / Roll Rate", fontsize=10, fontweight="bold", color="#404040")
        ax.set_xlabel("Roll rate  (°/s)", fontsize=8)
        ax.set_ylabel("n_z  (g)", fontsize=8)
        ax.set_xlim(-max_roll_deg, max_roll_deg)
        ax.set_ylim(MIN_NZ_G, MAX_NZ_G)
        ax.invert_yaxis()  # positive n_z downward (pilot convention)
        ax.axvline(0.0, color="#aaaaaa", linewidth=0.8)
        ax.axhline(1.0, color="#aaaaaa", linewidth=0.8)  # 1 g neutral
        ax.tick_params(labelsize=7)
        ax.set_facecolor("#fafafa")
        self._scatter_right = ax.scatter(
            [0.0], [1.0],
            s=80, color="#1565C0", zorder=5, clip_on=True,
        )

    def _tick(self) -> None:
        """Timer callback: drain queue, take latest frame, redraw."""
        frame = None
        while True:
            try:
                line = self._q.get_nowait()
                frame = json.loads(line)
            except queue.Empty:
                break
            except json.JSONDecodeError:
                continue

        if frame is not None:
            self._last_frame = frame

        self._redraw(self._last_frame)

        # Check for subprocess exit.
        if self._proc.poll() is not None:
            self._timer.stop()
            self._status.setText("joystick_verify exited.")

    def _redraw(self, frame: dict) -> None:
        nz          = frame["n_z"]
        ny          = frame["n_y"]
        roll_rps    = frame["roll_rate_wind_rps"]
        throttle    = frame["throttle_nd"]
        connected   = frame["connected"]
        roll_deg    = math.degrees(roll_rps)

        # Update 2D bug positions
        import numpy as np
        self._scatter_left.set_offsets(np.array([[ny, throttle]]))
        self._scatter_right.set_offsets(np.array([[roll_deg, nz]]))

        # Update bar gauges
        gauge_specs = [
            (nz,           MIN_NZ_G,   MAX_NZ_G,    1.0,               "n_z",       "g"),
            (ny,          -MAX_NY_G,   MAX_NY_G,    0.0,               "n_y",       "g"),
            (roll_deg,    -math.degrees(MAX_ROLL_RATE_RADS),
                           math.degrees(MAX_ROLL_RATE_RADS), 0.0,      "Roll rate",  "°/s"),
            (throttle * 100.0, 0.0,   100.0,        IDLE_THROTTLE_ND * 100.0,
                                                               "Throttle",  "%"),
        ]
        for ax, (val, vmin, vmax, vneutral, label, unit) in zip(self._ax_gauges, gauge_specs):
            _draw_gauge(ax, val, vmin, vmax, vneutral, label, unit)

        status_color  = "#1565C0" if connected else "#B71C1C"
        status_text   = "CONNECTED" if connected else "DISCONNECTED"
        self._status.setText(f"Joystick: {status_text}")
        self._status.setStyleSheet(
            f"padding-left: 6px; font-size: 11px; color: {status_color};"
        )

        self._canvas.draw_idle()

    def closeEvent(self, event) -> None:  # noqa: N802
        self._timer.stop()
        self._proc.terminate()
        self._proc.wait()
        super().closeEvent(event)

---
## Run

Run this cell to open the monitor window. Close the window to stop `joystick_verify`
and return to the notebook.

If no joystick is detected, `joystick_verify` will print an error to stderr and exit;
the window will not open.

In [ ]:
app = QApplication.instance() or QApplication(sys.argv)
window = InputMonitorWindow()
window.show()
app.exec()